#### 2026-02-19_accuracy_other_crops_2022.ipynb

**Author:** James Sayre  
**Email:** jsayre@ucdavis.edu  
**Date Modified:** 2026-02-19

**Description:** Computes accuracy metrics (standard R², within-R², between-R², RMSE) comparing AEF-based yield predictions for non-maize crops (sorghum, sugar, wheat, avocados) against INEGI 2022 agricultural census data and SIAP municipal estimates. Results are formatted as LaTeX tables for publication.

**Inputs:**
- `Data/predictions/adc_alpha_earth_preds_{crop}.csv` — AEF yield predictions per crop
- `Data/INEGI/MD_lab_outputs/LM2304-CA22-2025-09-29-superficie_ENTREGA/adc_land_use_ca22_adc07.dta` — INEGI 2022 census
- `The Promise of Crop Substitution/data/SIAP/Cleaned/siap_ag_prod_estimation_ca2007.dta` — SIAP municipal yields

**Outputs:**
- `plots/accuracy_other_crops_adc_2022.tex` — ADC-level accuracy table
- `plots/accuracy_other_crops_mun_2022.tex` — Municipality-level accuracy table

In [ ]:
import pandas as pd
import numpy  as np
import os

# ── Directories ──────────────────────────────────────────
home_dir      =  os.path.expanduser("~")
dropbox_dir   =  os.path.join(home_dir, "Dropbox", "Projects")
poppy_dir     =  os.path.join(dropbox_dir, "Maize_prediction")
crop_sub_dir  =  os.path.join(dropbox_dir, "The Promise of Crop Substitution")
siap_dir      =  os.path.join(crop_sub_dir, "data", "SIAP", "Cleaned")
pred_dir      =  os.path.join(poppy_dir, "Data", "predictions")
py_md_lab_dir =  os.path.join(poppy_dir, "Data", "INEGI", "MD_lab_outputs")
ca2022_dir    =  os.path.join(py_md_lab_dir, "LM2304-CA22-2025-09-29-superficie_ENTREGA")
plot_dir      =  os.path.join(poppy_dir, "plots")

# ── Inputs ─────────────────────────────────────────────
adc_gt_path   =  os.path.join(ca2022_dir, "adc_land_use_ca22_adc07.dta")      # INEGI 2022 census
siap_path     =  os.path.join(siap_dir, "siap_ag_prod_estimation_ca2007.dta")  # SIAP municipal yields

# ── Outputs ────────────────────────────────────────────
adc_tex_path  =  os.path.join(plot_dir, "accuracy_other_crops_adc_2022.tex")   # ADC-level table
mun_tex_path  =  os.path.join(plot_dir, "accuracy_other_crops_mun_2022.tex")   # Mun-level table

#### Metric Functions

In [ ]:
def standard_r2(y, yhat):
    """Standard R-squared: 1 - SS_res / SS_tot."""
    mask =  np.isfinite(y) & np.isfinite(yhat)
    y, yhat =  np.array(y)[mask], np.array(yhat)[mask]
    if len(y) < 2:
        return np.nan
    ss_res =  np.sum((y - yhat)**2)
    ss_tot =  np.sum((y - np.mean(y))**2)
    if ss_tot == 0:
        return np.nan
    return 1 - ss_res / ss_tot


def compute_rmse(y, yhat):
    """Root Mean Squared Error."""
    mask =  np.isfinite(y) & np.isfinite(yhat)
    y, yhat =  np.array(y)[mask], np.array(yhat)[mask]
    if len(y) == 0:
        return np.nan
    return np.sqrt(np.mean((y - yhat)**2))


def between_r2(df, y_col, yhat_col, group_col='muncode'):
    """Between-group R-squared: R-squared on group means."""
    sub =  df[[y_col, yhat_col, group_col]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(sub) == 0:
        return np.nan
    grp =  sub.groupby(group_col)[[y_col, yhat_col]].mean()
    return standard_r2(grp[y_col], grp[yhat_col])


def within_r2(df, y_col, yhat_col, group_col='muncode'):
    """Within-group R-squared: R-squared on deviations from group means."""
    sub =  df[[y_col, yhat_col, group_col]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if len(sub) == 0:
        return np.nan
    grp_counts =  sub.groupby(group_col).size()
    valid_grps =  grp_counts[grp_counts >= 2].index
    sub        =  sub[sub[group_col].isin(valid_grps)]
    if len(sub) == 0:
        return np.nan
    grp_means   =  sub.groupby(group_col)[[y_col, yhat_col]].transform('mean')
    y_w         =  sub[y_col] - grp_means[y_col]
    yhat_w      =  sub[yhat_col] - grp_means[yhat_col]
    return standard_r2(y_w, yhat_w)


def compute_all_metrics(df, y_col, yhat_col, group_col='muncode'):
    """Compute all accuracy metrics for a given prediction column."""
    sub =  df[[y_col, yhat_col, group_col]].replace([np.inf, -np.inf], np.nan).dropna()
    mask =  np.isfinite(sub[y_col]) & np.isfinite(sub[yhat_col])
    return {
        'N':          int(mask.sum()),
        'R2':         standard_r2(sub[y_col], sub[yhat_col]),
        'Between_R2': between_r2(sub, y_col, yhat_col, group_col),
        'Within_R2':  within_r2(sub, y_col, yhat_col, group_col),
        'RMSE':       compute_rmse(sub[y_col], sub[yhat_col]),
    }

#### Load Data

In [ ]:
### Load INEGI 2022 census
adc_gt =  pd.read_stata(adc_gt_path)
print(f"Census rows: {len(adc_gt):,}")

### Load SIAP municipal yields (2022)
siap_df              =  pd.read_stata(siap_path)
siap_df['yield_siap'] =  siap_df['q'] / siap_df['ha_planted']
siap_df['muncode']   =  siap_df['muncode'].apply(lambda x: str(int(x)).zfill(5))
siap_df              =  siap_df[siap_df['year'] == 2022]

### Define crops
crops = {
    'Sorghum':  {'pred_col': 'yield_pred_sorghum'},
    'Sugar':    {'pred_col': 'yield_pred_sugar'},
    'Wheat':    {'pred_col': 'yield_pred_wheat'},
    'Avocados': {'pred_col': 'yield_pred_avocados'},
}

#### Compute Accuracy Metrics

In [ ]:
all_adc_results         =  []
all_mun_results_census  =  []
all_mun_results_siap    =  []

for crop_name, info in crops.items():
    pred_col =  info['pred_col']
    pred_file =  os.path.join(pred_dir, f'adc_alpha_earth_preds_{crop_name.lower()}.csv')

    print(f"\n{'='*70}")
    print(f"  {crop_name}")
    print(f"{'='*70}")

    ### Load predictions (year=2022)
    pred_df        =  pd.read_csv(pred_file)
    pred_df        =  pred_df[pred_df['year'] == 2022]
    pred_df['adc'] =  pred_df['adcid'].str.replace('-', '', regex=False)
    pred_df        =  pred_df[['adc', pred_col]]

    ### Ground truth
    gt_crop =  adc_gt[adc_gt['name'] == crop_name].copy()

    ### Merge
    df =  gt_crop[['adc', 'muncode', 'land_input', 'vol_output', 'yield']].merge(
        pred_df, on='adc', how='left')

    ### SIAP
    siap_crop =  siap_df[siap_df['name'] == crop_name][['muncode', 'yield_siap']].copy()
    df        =  df.merge(siap_crop, on='muncode', how='left')

    print(f"  Census ADCs: {len(gt_crop):,}")
    print(f"  Matched:     {df[pred_col].notna().sum():,}")
    print(f"  SIAP:        {df['yield_siap'].notna().sum():,}")

    ### ---- Ex-post additive correction ----
    corr_col =  pred_col + '_corr'

    df['_wQ'] =  df[pred_col] * df['land_input']
    df['_wA'] =  df.apply(
        lambda x, col=pred_col: x['land_input'] if np.isfinite(x[col]) else 0, axis=1)

    mun_corr =  df.groupby('muncode')[['_wQ', '_wA']].sum().reset_index()
    mun_corr['pred_mun_avg'] =  mun_corr['_wQ'] / mun_corr['_wA']
    mun_corr.loc[mun_corr['_wA'] == 0, 'pred_mun_avg'] = np.nan
    mun_corr =  mun_corr.merge(siap_crop, on='muncode', how='left')
    mun_corr['diff'] =  mun_corr['pred_mun_avg'] - mun_corr['yield_siap']

    df =  df.merge(mun_corr[['muncode', 'diff']], on='muncode', how='left')
    df[corr_col] =  (df[pred_col] - df['diff']).clip(lower=0)
    df.loc[df[pred_col].isna(), corr_col] = np.nan
    print(f"  Corrected:   {df[corr_col].notna().sum():,}")

    ### ---- ADC-level metrics ----
    for col, model_lbl in [(pred_col, 'AEF mean'), (corr_col, 'AEF mean Corr.'), ('yield_siap', 'SIAP')]:
        m        =  compute_all_metrics(df, 'yield', col)
        m['Model'] =  model_lbl
        m['Crop']  =  crop_name
        all_adc_results.append(m)
        print(f"  {model_lbl:12s}  N={m['N']:>6,}  R2={m['R2']:.3f}  "
              f"Btw={m['Between_R2']:.3f}  Wtn={m['Within_R2']:.3f}  RMSE={m['RMSE']:.3f}")

    ### ---- Municipality-level: aggregate predictions ----
    df['_wQ_c'] =  df[corr_col] * df['land_input']
    df['_wA_c'] =  df.apply(
        lambda x: x['land_input'] if np.isfinite(x[corr_col]) else 0, axis=1)

    mun_agg =  df.groupby('muncode').agg({
        '_wQ': 'sum', '_wA': 'sum', '_wQ_c': 'sum', '_wA_c': 'sum',
        'vol_output': 'sum', 'land_input': 'sum', 'yield_siap': 'first',
    }).reset_index()

    mun_agg['yield_mun_census']  =  mun_agg['vol_output'] / mun_agg['land_input']
    mun_agg['pred_mun_raw']      =  mun_agg['_wQ'] / mun_agg['_wA']
    mun_agg.loc[mun_agg['_wA'] == 0, 'pred_mun_raw'] = np.nan
    mun_agg['pred_mun_corr']     =  mun_agg['_wQ_c'] / mun_agg['_wA_c']
    mun_agg.loc[mun_agg['_wA_c'] == 0, 'pred_mun_corr'] = np.nan

    ### vs INEGI census
    print(f"\n  Municipality-level vs Census:")
    for col, lbl in [('pred_mun_raw', 'AEF mean (agg.)'), ('pred_mun_corr', 'AEF mean Corr. (agg.)'),
                     ('yield_siap', 'SIAP Mun. Avg.')]:
        sub  =  mun_agg[['yield_mun_census', col]].replace([np.inf,-np.inf], np.nan).dropna()
        r2v  =  standard_r2(sub['yield_mun_census'], sub[col])
        rmsv =  compute_rmse(sub['yield_mun_census'], sub[col])
        all_mun_results_census.append({'Crop': crop_name, 'Model': lbl, 'N': len(sub), 'R2': r2v, 'RMSE': rmsv})
        print(f"    {lbl:20s}  N={len(sub):>5,}  R2={r2v:.3f}  RMSE={rmsv:.3f}")

    ### vs SIAP
    print(f"  Municipality-level vs SIAP:")
    for col, lbl in [('pred_mun_raw', 'AEF mean (agg.)'), ('yield_mun_census', 'Census (agg.)')]:
        sub  =  mun_agg[['yield_siap', col]].replace([np.inf,-np.inf], np.nan).dropna()
        r2v  =  standard_r2(sub['yield_siap'], sub[col])
        rmsv =  compute_rmse(sub['yield_siap'], sub[col])
        all_mun_results_siap.append({'Crop': crop_name, 'Model': lbl, 'N': len(sub), 'R2': r2v, 'RMSE': rmsv})
        print(f"    {lbl:20s}  N={len(sub):>5,}  R2={r2v:.3f}  RMSE={rmsv:.3f}")

#### Write LaTeX Tables

In [ ]:
### ------------------------------------------------------------------ ###
### ADC-level table
### ------------------------------------------------------------------ ###

lines =  []
lines.append(r"\begin{table}[htbp]")
lines.append(r"\centering")
lines.append(r"\caption{ADC-level yield prediction accuracy for non-maize crops vs.\ INEGI 2022 census}")
lines.append(r"\label{tab:accuracy_other_crops_adc}")
lines.append(r"\begin{tabular}{llrrrrr}")
lines.append(r"\hline")
lines.append(r"Crop & Model & $N$ & $R^2$ & Between $R^2$ & Within $R^2$ & RMSE \\")
lines.append(r"\hline")

for r in all_adc_results:
    r2_s   =  f"{r['R2']:.3f}"         if np.isfinite(r['R2'])         else "---"
    btw_s  =  f"{r['Between_R2']:.3f}" if np.isfinite(r['Between_R2']) else "---"
    wtn_s  =  f"{r['Within_R2']:.3f}"  if np.isfinite(r['Within_R2'])  else "---"
    rmse_s =  f"{r['RMSE']:.3f}"       if np.isfinite(r['RMSE'])       else "---"
    lines.append(f"{r['Crop']} & {r['Model']} & {r['N']:,} & {r2_s} & {btw_s} & {wtn_s} & {rmse_s} \\\\")
    if r['Model'] == 'SIAP':
        lines.append(r"\hline")

lines.append(r"\end{tabular}")
lines.append(r"\end{table}")

tex_adc =  "\n".join(lines)
with open(adc_tex_path, 'w') as f:
    f.write(tex_adc + "\n")
print(f"Written: {adc_tex_path}")
print(tex_adc)

In [ ]:
### ------------------------------------------------------------------ ###
### Municipality-level table
### ------------------------------------------------------------------ ###

lines =  []
lines.append(r"\begin{table}[htbp]")
lines.append(r"\centering")
lines.append(r"\caption{Municipality-level yield prediction results for non-maize crops, 2022. "
             r"Predictions are aggregated from the ADC level using planted-area weights.}")
lines.append(r"\label{tab:mun_other_crops}")
lines.append(r"\begin{tabular}{llrrr}")
lines.append(r"\hline")
lines.append(r"\multicolumn{5}{l}{\textit{Panel A: vs.\ INEGI Census (aggregated)}} \\")
lines.append(r"\hline")
lines.append(r"Crop & Model & $N$ & $R^2$ & RMSE \\")
lines.append(r"\hline")

for r in all_mun_results_census:
    r2_s   =  f"{r['R2']:.3f}" if np.isfinite(r['R2']) else "---"
    rmse_s =  f"{r['RMSE']:.3f}" if np.isfinite(r['RMSE']) else "---"
    model_s =  r['Model'].replace('Avg.', 'Avg.\\ ')
    lines.append(f"{r['Crop']} & {r['Model']} & {r['N']:,} & {r2_s} & {rmse_s} \\\\")

lines.append(r"\hline")
lines.append(r"\multicolumn{5}{l}{\textit{Panel B: vs.\ SIAP Municipal Estimates}} \\")
lines.append(r"\hline")
lines.append(r"Crop & Model & $N$ & $R^2$ & RMSE \\")
lines.append(r"\hline")

for r in all_mun_results_siap:
    r2_s   =  f"{r['R2']:.3f}" if np.isfinite(r['R2']) else "---"
    rmse_s =  f"{r['RMSE']:.3f}" if np.isfinite(r['RMSE']) else "---"
    lines.append(f"{r['Crop']} & {r['Model']} & {r['N']:,} & {r2_s} & {rmse_s} \\\\")

lines.append(r"\hline")
lines.append(r"\end{tabular}")
lines.append(r"\end{table}")

tex_mun =  "\n".join(lines)
with open(mun_tex_path, 'w') as f:
    f.write(tex_mun + "\n")
print(f"Written: {mun_tex_path}")
print(tex_mun)